# 10 — LangGraph Persistence, Interrupt/HITL, Time Travel & Subgraphs

## Learning requirements
- checkpoint/thread;
- durable resume;
- `interrupt()` + `Command(resume=...)`;
- replay/fork/time travel;
- subgraph composition;
- idempotency khi resume/retry side-effect.

Durable execution không chỉ là “memory”; nó cho phép workflow sống qua process crash và human wait.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class ApprovalState(TypedDict):
    action: str
    approved: bool

def request_approval(state: ApprovalState):
    answer = interrupt({
        "type": "approval",
        "action": state["action"],
        "question": f"Approve action: {state['action']}?"
    })
    return {"approved": bool(answer)}

def execute(state: ApprovalState):
    if not state["approved"]:
        return {}
    # Demo only. Production side effects must be idempotent.
    print("EXECUTED:", state["action"])
    return {}

b = StateGraph(ApprovalState)
b.add_node("approval", request_approval)
b.add_node("execute", execute)
b.add_edge(START, "approval")
b.add_edge("approval", "execute")
b.add_edge("execute", END)

approval_graph = b.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "approval-001"}}

paused = approval_graph.invoke(
    {"action": "send_customer_email", "approved": False},
    config,
)
print(paused)

In [ ]:
# Resume after a human/system sends a decision.
resumed = approval_graph.invoke(Command(resume=True), config)
print(resumed)

## Critical idempotency rule

Khi node có:
- payment,
- email,
- database mutation,
- external API write,

workflow retry/resume có thể execute lại code nếu bạn thiết kế sai.

Side-effect phải có idempotency key hoặc được tách sao cho retry an toàn.

## Time travel exercise

1. Chạy graph qua nhiều checkpoints.
2. Inspect state history.
3. Replay từ checkpoint cũ.
4. Fork với state khác.
5. Ghi lại sự khác biệt giữa replay và new thread.

## Subgraph exercise

Tạo:
```text
Parent Interview Graph
  |- Code Scan Subgraph
  |- Questioning Subgraph
  `- Validation Subgraph
```

Xác định subgraph nào cần own persistence và state nào được share.

## Required output
- approval demo;
- replay/fork demo;
- 2+ subgraphs;
- note về idempotency.

## Done criteria
Bạn biết cách pause hàng giờ/ngày và resume từ persisted state mà không rerun toàn workflow.